In [ ]:
#| default_exp data_preprocessing

# Data Preprocessing

> Dont forget, none of this matters

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import zarr, numpy as np, pandas as pd, multiprocessing as mp, torch, warnings, tqdm
from functools import partial
from wavefm.signal import stft, resample_waveform
from collections import Counter

In [ ]:
#| export
def calculate_class_weights(dataloader, ignore_index=-100, returns_padded_mask=True, return_ratio=True):
    if returns_padded_mask:
        target_values = [y.unique(return_counts=True) for _,y,_ in dataloader]
    else:
        target_values = [y.unique(return_counts=True) for _,y in dataloader]
    all_counts = []
    for i,c in target_values:
        if ignore_index is not None:
            all_counts.append(c[i != ignore_index])
        else:
            all_counts.append(c)
    idx = i[i != ignore_index] if ignore_index is not None else i
    total_per_class = torch.stack(all_counts).sum(dim=0)
    total = total_per_class.sum()
    if return_ratio:
        ratio = total_per_class/total
    else:
        ratio = total_per_class
    return dict(zip(idx.tolist(), ratio))


In [ ]:
#| export
def interpolate_nan_clip(x, physiological_range_clip=None, percentile_clip=None, return_mask_only=False):
    """
    Function to clip outliers based on percentiles or physiological range and then interpolate nearby values
    """
    if physiological_range_clip is not None:
        # if a physiological range, clip, set to nan, and interpolate nearby values
        assert len(physiological_range_clip) == 2, "physiological_range_clip expects a tuple or list of 1 or 2 values. Supply none to clip only one end."
        max_ = physiological_range_clip[1] if physiological_range_clip[1] is not None else None
        min_ = physiological_range_clip[0] if physiological_range_clip[0] is not None else None
        if max_ is not None:
            x[x>=max_] = np.nan
        if min_ is not None:
            x[x<=min_] = np.nan
    if percentile_clip is not None:
        # if percentiles, clip at percentiles, set to nana, and interpolate nearby
        assert len(percentile_clip) == 2, "percentile_clip expects a tuple or list of 1 or 2 values. Supply none to clip only one end."
        max_ = np.quantile(x, q=percentile_clip[1]) if percentile_clip[1] is not None else None
        min_ = np.quantile(x, q=percentile_clip[0]) if percentile_clip[0] is not None else None
        if max_ is not None:
            x[x>=max_] = np.nan
        if min_ is not None:
            x[x<=min_] = np.nan
    mask = np.isnan(x) ^ np.isinf(x) 
    if return_mask_only:
        return mask
    if np.all(mask):
        x = np.zeros_like(x) # return all zeros if all nan
    elif np.any(mask):
        x[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), x[~mask])
    return x

In [ ]:
#| export
def calculate_stats(idx, zarr_file, channels, clip_interpolations=None, channel_magnitude_multiple=None):
    """
    Function to caluclate stats on an individual zarr array, including a clip interpolate range
    """
    root_grp = zarr.open(zarr_file)
    updated_channels = []
    if any(isinstance(i, list) for i in channels):
        avail_channels = list(root_grp.array_keys())
        for p in channels:
            updated_channels.append(next(x for x in p if x in avail_channels))
    else:
        updated_channels = channels
    channel_tracker = {channel:{'std':0,'len':0,'mean':0, 'n':0, 'n_zeros':0, 'n_nan':0, '50%':0, '95%':0, '99%':0, '5%':0, '1%':0} for channel in updated_channels}
    for channel in updated_channels:
        if channel in root_grp.array_keys():
            channel_tracker[channel]['n'] += 1
            temp = root_grp[channel][:].astype(np.float32)
            if clip_interpolations is not None and clip_interpolations != {} and channel in clip_interpolations:
                temp = interpolate_nan_clip(temp, physiological_range_clip=clip_interpolations[channel]['phys_range'], percentile_clip=clip_interpolations[channel]['percentiles'])
            channel_tracker[channel]['n_zeros'] = (temp == 0).sum()
            channel_tracker[channel]['n_nan'] = np.isnan(temp).sum()
            channel_tracker[channel]['file'] = zarr_file
            channel_tracker[channel]['mean'] += np.nanmean(temp)
            channel_tracker[channel]['len'] += len(temp)
            channel_tracker[channel]['std'] += np.nanstd(temp)
            quantiles = np.nanquantile(temp, q=[0.01,0.05,0.5,0.95,0.99])
            for idx, quantile in enumerate(['1%','5%','50%', '95%','99%']):
                channel_tracker[channel][quantile] += quantiles[idx]
    if channel_magnitude_multiple is not None:
        for channel in updated_channels:
            if channel in channel_magnitude_multiple:
                for stat in ['mean', 'std', '1%','5%','50%', '95%','99%']:
                    channel_tracker[channel][stat] *= channel_magnitude_multiple[channel]
    return channel_tracker

def calculate_stats_all(zarr_files, channels, sample_wise=True, clip_interpolations=None, channel_magnitude_multiple=None):
    with mp.Pool() as pool:
        f = partial(calculate_stats, channels=channels, clip_interpolations=clip_interpolations, channel_magnitude_multiple=channel_magnitude_multiple)
        results = pool.starmap_async(f, enumerate(zarr_files))
        pool.close()
        pool.join()
    if not sample_wise:
        all_channels = [v for i in channels for v in i]
        total_stats = {channel:{'std':0,'len':0,'mean':0, 'n':1, 'n_zeros':0, 'n_nan':0, '50%':0, '95%':0, '99%':0, '5%':0, '1%':0} for channel in all_channels}
        for result in results.get():
            for channel in result:
                for stat in ['n','len','n_zeros','n_nan', 'std','mean','1%','5%','50%','95%','99%']:
                    total_stats[channel][stat] += result[channel][stat]
        for channel in total_stats:
            for stat in ['std','mean','1%','5%','50%','95%','99%']:
                total_stats[channel][stat] = total_stats[channel][stat] / total_stats[channel]['n']
        return pd.DataFrame(total_stats).T
    else:
        final_df = pd.DataFrame()
        for result in results.get():
            final_df = pd.concat([final_df, pd.DataFrame(result).T])
        return final_df

In [ ]:
#| export
def calculate_stft_stats(idx, 
                         zarr_file, # zarr file with arrays
                         channels, # channels to perform stft and calculate stats on
                         n_fft=256, # number of ffts to perform
                         win_length=(2*125), # window of ffts
                         pad_mode='replicate', # padding mode
                         center=True, # center stft?
                         hop_length=(125), # hop length?
                         normalized=True,
                         decibel_scale=True,
                         return_complex=True,
                         clip_interpolations=None):
    """
    Function to calculate stft stats on an individual zarr array, including a clip interpolate range
    """
    channel_tracker = {channel:{'std':0,'mean':0, 'n':0, 'n_zeros':0, '50%':0, '95%':0, '99%':0, '5%':0, '1%':0} for channel in channels}
    root_grp = zarr.open(zarr_file)
    signals = []
    for channel in channels:
        if channel in root_grp.array_keys():
            temp = root_grp[channel][:].astype(np.float32)
            if clip_interpolations is not None and clip_interpolations != {} and channel in clip_interpolations:
                temp = interpolate_nan_clip(temp, physiological_range_clip=clip_interpolations[channel]['phys_range'], percentile_clip=clip_interpolations[channel]['percentiles'])
            signals.append(temp)
    # [bs x n_vars x n_fft // 2 + 1 x stft_len]
    stfts = stft(np.stack(signals), n_fft=n_fft, win_length=win_length, pad_mode=pad_mode, center=center, hop_length=hop_length, normalized=normalized, decibel_scale=decibel_scale, return_complex=return_complex)
    q = torch.tensor([0.01,0.05,0.5,0.95,0.99])
    for i, channel in enumerate(channels):
        channel_tracker[channel]['n'] += 1
        channel_tracker[channel]['mean'] += stfts[...,i,:,:].mean().item()
        channel_tracker[channel]['std'] += stfts[...,i,:,:].std().item()
        channel_tracker[channel]['n_zeros'] = (stfts[...,i,:,:] == 0).sum().item()
        channel_tracker[channel]['file'] = zarr_file
        quantiles = torch.quantile(stfts[...,i,:,:], q, keepdim=False)
        for idx, quantile in enumerate(['1%','5%','50%', '95%','99%']):
             channel_tracker[channel][quantile] += quantiles[idx].item()
    return channel_tracker

def calculate_stft_stats_batch(batch, 
                               n_fft=256, # number of ffts to perform
                               win_length=(2*125), # window of ffts
                               pad_mode='replicate', # padding mode
                               center=True, # center stft?
                               hop_length=(125), # hop length?
                               normalized=True,
                               decibel_scale=True,
                               return_complex=True):
    stfts = stft(batch, n_fft=n_fft, win_length=win_length, pad_mode=pad_mode, center=center, hop_length=hop_length, normalized=normalized, channel_stft_means=None, channel_stft_stds=None, decibel_scale=decibel_scale, return_complex=return_complex)
    means = stfts.mean(dim=[0,-2,-1], keepdim=True)
    stds = stfts.std(dim=[0,-2,-1], keepdim=True)
    return means, stds

In [ ]:
#| export
def check_signal_not_constant(channel_data, constant_tolerance=1.0):
    """
    Function to check if a signal is NOT constant. This works for NaNs and constant values.
    """
    if len(channel_data) == 0:
        return False
    _, counts = np.unique(channel_data, return_counts=True)
    max_ratio = np.max(counts) / len(channel_data)
    return max_ratio <= constant_tolerance # if more than half the signal is the same value, return False
    
def calculate_samples(zarr_file, channels, frequency, sample_seq_len_sec, stride_sec, start_offset_sec=None, max_seq_len_sec=None, include_partial_samples=True, require_all_channels=True, constant_nan_tolerance=1.0, min_seq_len_sec=None, constant_channels=None):
    """
    Function to create a dataframe of samples and their sequence indices
    """
    if start_offset_sec is None:
        start_offset_sec = 0
    if min_seq_len_sec is None:
        min_seq_len_sec = 0
    if constant_channels is None:
        constant_channels = []
    start_offset = start_offset_sec * frequency
    if max_seq_len_sec is not None:
        assert max_seq_len_sec >= sample_seq_len_sec, "The maximum sequence length should be >= the sample sequence length. The maximum sequence length is the end cutoff point of a sample. A sample cannot be longer than that value."
    if min_seq_len_sec is not None and max_seq_len_sec is not None:
        assert min_seq_len_sec <= max_seq_len_sec, "The minimum sequence length should be <= the max sequence length. The minimum sequence length is the start cutoff point of a sample. A sample cannot be shorter than that value."
    sample_seq_len = sample_seq_len_sec * frequency

    stride = stride_sec * frequency
    root_grp = zarr.open(zarr_file)
    updated_channels = []
    avail_channels = list(root_grp.array_keys())
    if all(isinstance(i, list) for i in channels):
        for p in channels:
            updated_channels.append(next((x for x in p if x in avail_channels), None))
    else:
        updated_channels = [p if p in avail_channels else None for p in channels]
    reason = 0
    if (None not in updated_channels) or (not require_all_channels and any(updated_channels)):
        # all channels are present or at least one channel is present and require_all_channels is False
        ## all channels should be the same length
        if 'header' in root_grp.attrs:
            duration = int(float(root_grp.attrs['header']['Duration'])) # duration in seconds
        else:
            duration = int(float(root_grp.attrs['Duration'])) # duration in seconds
        if duration == -1:
            warnings.warn(f"Duration is -1 for file {zarr_file}. Inferring from signal...")
            test_channel = next((i for i in updated_channels if i is not None))
            signal_frequency = root_grp[test_channel].attrs.asdict().get('signal_header', None).get('sample_frequency', None)
            if signal_frequency is None:
                duration = len(root_grp[test_channel][:]) // frequency
            else:
                duration = len(root_grp[test_channel][:]) // signal_frequency
        if duration > (start_offset_sec + min_seq_len_sec):
            if include_partial_samples:
                # if max_seq_len_sec is not None, use that, otherwise use the duration
                max_seq_len = max_seq_len_sec*frequency if max_seq_len_sec is not None else duration*frequency+sample_seq_len-1
            else:
                max_seq_len =  max_seq_len_sec*frequency if max_seq_len_sec is not None and max_seq_len_sec < duration else duration*frequency
            sample_indices = [(i, i+sample_seq_len) 
                    for i in range(start_offset, max_seq_len+start_offset, stride) 
                    if (i-start_offset)+sample_seq_len <= max_seq_len]
            sample_index_df = pd.DataFrame([{'start_idx':i[0], 'end_idx':i[1]} for i in sample_indices])
            sample_index_df['file'] = zarr_file
            if not sample_index_df.empty:
                exclude_channels = []
                for channel in updated_channels:
                    if channel is not None and channel not in constant_channels:
                        channel_data = root_grp[channel][:]
                        signal_header = root_grp[channel].attrs.asdict()#.get('signal_header', None)
                        if 'signal_header' in signal_header:
                            # for sleep files
                            signal_header = signal_header['signal_header']
                            channel_frequency = signal_header.get('sample_frequency', None)
                        elif 'sampling_frequency' in signal_header:
                            # for bedmaster files
                            channel_frequency = signal_header['sampling_frequency']
                        else:
                            warnings.warn(f"Channel {channel} has no signal header for file {zarr_file}. Skipping missingness and constant checks.")
                            exclude_channels.append(channel)
                            continue
                        if channel_frequency is None:
                            warnings.warn(f"Channel {channel} has no sample frequency for file {zarr_file}. Setting to passed frequency {frequency}.")
                            channel_frequency = frequency
                        sample_index_df['channel_frequency'] = channel_frequency
                        sample_index_df['channel_frequency_start_idx'] = (sample_index_df['start_idx'] / frequency * channel_frequency).astype(int)
                        sample_index_df['channel_frequency_end_idx'] = (sample_index_df['end_idx'] / frequency * channel_frequency).astype(int)

                        start_indices = sample_index_df['channel_frequency_start_idx'].values
                        end_indices = sample_index_df['channel_frequency_end_idx'].values
                        not_constant_results = []

                        for start_idx, end_idx in zip(start_indices, end_indices):
                            segment = channel_data[start_idx:end_idx]
                            not_constant_results.append(check_signal_not_constant(segment, constant_tolerance=constant_nan_tolerance))
                        
                        sample_index_df[f'{channel}_not_constant_nan'] = not_constant_results
                        sample_index_df.drop(columns=['channel_frequency_start_idx', 'channel_frequency_end_idx', 'channel_frequency'], inplace=True)
                # the maximum channel nan sum should be less than the nan tolerance
                sample_index_df['all_channels_not_constant_nan'] = sample_index_df[[f'{channel}_not_constant_nan' for channel in updated_channels if channel is not None and channel not in exclude_channels and channel not in constant_channels]].all(axis=1)
                sample_index_df = sample_index_df.loc[(sample_index_df['all_channels_not_constant_nan'] == True)]
                if sample_index_df.empty:
                    # all constant
                    return None, 4        
                sample_index_df['n_samples'] = len(sample_indices)
                return sample_index_df, reason
            else:
                # empty df
                return None, 3
        else:
            # start after duration
            return None, 2
    else:
        # missing channels
        return None, 1

def calculate_samples_mp(zarr_files, channels, frequency, sample_seq_len_sec, stride_sec, start_offset_sec = None, max_seq_len_sec=None, include_partial_samples=True, constant_nan_tolerance=1.0, require_all_channels=True, min_seq_len_sec=None, constant_channels=None):
    """
    Multiprocessing function to generate samples
    """
    final_df = pd.DataFrame(columns=['file', 'start_idx','end_idx','n_samples'])
    reason_tracker = {0:'valid', 1:'missing channels', 2:'start after duration', 3:'empty df', 4:'nan filters'}
    reasons = []
    total_samples = 0
    with mp.Pool() as pool:
        f = partial(calculate_samples, channels=channels, frequency=frequency, start_offset_sec=start_offset_sec, max_seq_len_sec=max_seq_len_sec, sample_seq_len_sec=sample_seq_len_sec, stride_sec=stride_sec, include_partial_samples=include_partial_samples, constant_nan_tolerance=constant_nan_tolerance, require_all_channels=require_all_channels, min_seq_len_sec=min_seq_len_sec, constant_channels=constant_channels)
        # Use tqdm to show progress
        results = list(tqdm.tqdm(
            pool.imap_unordered(f, zarr_files),
            total=len(zarr_files),
            desc="Processing files"
        ))
        pool.close()
        pool.join()
    for result, reason in results:
        final_df = pd.concat([final_df, result])
        reasons.append(reason)
    final_df.reset_index(drop=True, inplace=True)
    total_samples = len(final_df)
    print('REMOVAL REASONS')
    print({reason_tracker[k]:v for k,v in Counter(reasons).items()})
    return final_df, total_samples

In [ ]:
#| notest
channels = [['ECG', 'ECG (LL-RA)', 'EKG', 'ECG (L-R)'],
            ['EOG(L)', 'EOG-L', 'E1', 'LOC', 'E1-M2', 'E1-AVG'],
            ['EMG', 'cchin_l', 'chin', 'EMG (L-R)', 'EMG (1-2)', 'EMG (1-3)', 'Chin3'],
            ['C4-M1', 'C4_M1', 'EEG', 'EEG3', 'C3-M2', 'C3_M2', 'C4-AVG'],
            ['SaO2', 'SpO2','spo2'],
            ['THOR RES', 'Thor', 'thorax', 'Thoracic', 'Chest'],
            ['ABDO RES', 'abdomen', 'Abdo', 'Abdominal', 'ABD']]
channels = ['ABP', 'II', 'V', 'PLETH','RESP']

updated_channels = []
avail_channels = ['II']
if all(isinstance(i, list) for i in channels):
        for p in channels:
            updated_channels.append(next((x for x in p if x in avail_channels), None))
else:
    updated_channels = [p if p in avail_channels else None for p in channels]
updated_channels

[None, 'II', None, None, None]

In [ ]:
#| export
def check_hypnogram(zarr_file, required_stages=[0,1,2,3,4], constant_tolerance=1.0):
    """
    Function to check if a hypnogram contains all required stages and is not constant
    """
    root_grp = zarr.open(zarr_file)
    if 'hypnogram' not in root_grp.array_keys():
        reason = 4
        return False, reason
    x = root_grp['hypnogram'][:]
    if len(x) == 0:
        reason = 1
        return False, reason
    unique, counts = np.unique(x, return_counts=True)
    stages_present = set(unique)
    stages_required = set(required_stages)
    if not stages_required.issubset(stages_present):
        reason = 2
        return False, reason
    max_ratio = np.max(counts) / len(x)
    if max_ratio > constant_tolerance:
        reason = 3
        return False, reason
    else:
        reason = 0
        return True, reason

def check_hypnograms_mp(zarr_files, required_stages=[0,1,2,3,4], constant_tolerance=1.0):
    """
    Multiprocessing function to check hypnograms
    """
    valid_files = []
    reason_tracker = {0:'valid', 1:'length 0', 2:'missing required stages', 3:'above constant tolerance', 4:'no hypnogram'}
    reasons = []
    with mp.Pool() as pool:
        f = partial(check_hypnogram, required_stages=required_stages, constant_tolerance=constant_tolerance)
        results = list(tqdm.tqdm(
            pool.imap_unordered(f, zarr_files),
            total=len(zarr_files),
            desc="Checking hypnograms"
        ))
        pool.close()
        pool.join()
    for zarr_file, result in zip(zarr_files, results):
        valid, reason = result
        reasons.append(reason)
        if valid:
            valid_files.append(zarr_file)
    print('HYPNOGRAM REMOVAL REASONS')
    print({reason_tracker[k]:v for k,v in Counter(reasons).items()})
    return valid_files

In [ ]:
#| export
def calculate_samples_forecast(zarr_file, outcome_start_times, outcome_durations, outcome_vals, channels, forecast_window_sec, frequency, sample_seq_len_sec, require_all_channels=True, channel_quality_functions=None, infer_forecast_windows=True, sample_frequency_key='sampling_frequency'):
    """
    Function to create a dataframe of samples and their sequence indices
    """
    if len(outcome_start_times) != len(outcome_vals):
        raise ValueError("outcome_start_times and outcome_vals must have same length")
    sample_seq_len = int(sample_seq_len_sec * frequency)
    outcome_start_idxs = [int(t * frequency) for t in outcome_start_times]
    outcome_end_idxs = [s + int(e * frequency) for s,e in zip(outcome_start_idxs,outcome_durations)]
    forecast_offsets = [int(f * frequency) for f in forecast_window_sec]
    
    root_grp = zarr.open(zarr_file, mode='r')

    avail_channels = list(root_grp.array_keys())
    removal_reasons = {}
    
    updated_channels = []
    for ch in channels:
        if isinstance(ch, list):
            # Find first available channel from alternatives
            selected = next((x for x in ch if x in avail_channels), None)
        else:
            selected = ch if ch in avail_channels else None
        updated_channels.append(selected)

    # Check channel requirements
    valid_channels = [ch for ch in updated_channels if ch is not None]
    if require_all_channels and len(valid_channels) != len(channels):
        #missing = [ch for ch in channels if ch not in avail_channels]
        #raise ValueError(f"Required channels not available: {missing}")
        removal_reasons['1'] = 1 # missing channels
        return None, removal_reasons
    elif not valid_channels:
        removal_reasons['2'] = 1 # no valid channels
        return None, removal_reasons
    
    sample_dfs = []
    for forecast_sec, forecast_offset in zip(forecast_window_sec, forecast_offsets):
        end_idxs = [outcome_idx - forecast_offset for outcome_idx in outcome_start_idxs]
        start_idxs = [end_idx - sample_seq_len for end_idx in end_idxs]
        df = pd.DataFrame({
            'start_idx': start_idxs,
            'end_idx': end_idxs,
            'file_path': zarr_file,
            'outcome_start_idx': outcome_start_idxs,
            'outcome_end_idx': outcome_end_idxs,
            'outcome_duration_sec': outcome_durations,
            f'outcome_val_{forecast_sec}sec': outcome_vals
        })
        sample_dfs.append(df)
    sample_index_df = pd.concat(sample_dfs, ignore_index=True)
    agg_dict = {col: 'first' for col in sample_index_df.columns 
                if col not in ['start_idx', 'end_idx', 'file_path']}
    sample_index_df = sample_index_df.groupby(['start_idx', 'end_idx', 'file_path']).agg(agg_dict).reset_index()
    total_before = len(sample_index_df)

    data_length = len(root_grp[valid_channels[0]])
    valid_mask = (sample_index_df['start_idx'] >= 0) & (sample_index_df['end_idx'] <= data_length)
    sample_index_df = sample_index_df.loc[valid_mask]
    removal_reasons['3'] = (total_before - len(sample_index_df)) # number of samples removed due to invalid indices
    
    if infer_forecast_windows:
        for forecast_offset, forecast_sec in zip(forecast_offsets, forecast_window_sec):
            # check if the forecast window is within the outcome window, else assign to 0
            ## note that to force a 0, end outcome idx must be less than start outcome idx
            forecast_start_idxs = sample_index_df['end_idx'] + forecast_offset
            #inferred_ = (forecast_start_idxs >= sample_index_df['outcome_start_idx']) & (forecast_start_idxs <= sample_index_df['outcome_end_idx'])
            sample_index_df[f'outcome_val_{forecast_sec}sec'] = (forecast_start_idxs >= sample_index_df['outcome_start_idx']) & (forecast_start_idxs <= sample_index_df['outcome_end_idx']).astype(int)

    if not sample_index_df.empty:
        for channel in updated_channels:
            if channel is not None:
                func = channel_quality_functions.get(channel, None) if channel_quality_functions is not None else None
                if func is None:
                    sample_index_df[f'{channel}_quality'] = [True] * len(sample_index_df)
                    continue
                channel_frequency = root_grp[channel].attrs.get(sample_frequency_key, None)
                if channel_frequency is None:
                    warnings.warn(f"Channel {channel} has no sampling frequency attribute in the zarr file. Setting to {frequency} Hz")
                    channel_frequency = frequency
                sample_index_df['channel_frequency'] = channel_frequency
                sample_index_df['channel_frequency_start_idx'] = (sample_index_df['start_idx'] / frequency * channel_frequency).astype(int)
                sample_index_df['channel_frequency_end_idx'] = (sample_index_df['end_idx'] / frequency * channel_frequency).astype(int)

                start_indices = sample_index_df['channel_frequency_start_idx'].values
                end_indices = sample_index_df['channel_frequency_end_idx'].values
                quality_results = []

                for start_idx, end_idx in zip(start_indices, end_indices):
                    segment = root_grp[channel][start_idx:end_idx]
                    quality_results.append(func(segment, fs=channel_frequency))
                sample_index_df[f'{channel}_quality'] = quality_results
                sample_index_df.drop(columns=['channel_frequency_start_idx', 'channel_frequency_end_idx', 'channel_frequency'], inplace=True)
        
        # the maximum channel nan sum should be less than the nan tolerance
        sample_index_df['all_channels_quality'] = sample_index_df[[f'{channel}_quality' for channel in updated_channels if channel is not None]].all(axis=1)
        total_before = len(sample_index_df)
        sample_index_df = sample_index_df.loc[(sample_index_df['all_channels_quality'] == True)]
        removal_reasons['5'] = (total_before - len(sample_index_df)) # number of samples removed due to nan or constant signal
                    
        sample_index_df['n_samples'] = 1
        del root_grp
        return sample_index_df, removal_reasons
    else:
        removal_reasons['6'] = 1 # no valid samples
        del root_grp
        return None, removal_reasons
        
def calculate_samples_forecast_mp(outcome_df, file_col, outcome_val_col, outcome_time_col, outcome_duration_col, forecast_window_sec, channels, frequency, sample_seq_len_sec, channel_quality_functions=None, require_all_channels=True, infer_forecast_windows=True, sample_frequency_key='sampling_frequency'):
    """
    Multiprocessing function to generate samples
    """
    #args = outcome_df.groupby(file_col, as_index=False).agg({outcome_time_col:list, outcome_duration_col:list, outcome_val_col:list})
    args = outcome_df[[file_col, outcome_time_col, outcome_duration_col, outcome_val_col]]
    for col in [outcome_time_col, outcome_duration_col, outcome_val_col]:
        args[col] = args[col].apply(lambda x: [x] if not isinstance(x, list) else x)
    args = args.to_dict(orient='records')
    args = list(map(lambda v: tuple(v.values()), args))
    removal_reasons = {"1":0, "2":0, "3":0, "4":0, "5":0}
    with mp.Pool() as pool:
        f = partial(calculate_samples_forecast, channels=channels, forecast_window_sec=forecast_window_sec, frequency=frequency, sample_seq_len_sec=sample_seq_len_sec, channel_quality_functions=channel_quality_functions, require_all_channels=require_all_channels, infer_forecast_windows=infer_forecast_windows, sample_frequency_key=sample_frequency_key)
        results = pool.starmap(f, tqdm.tqdm(args, total=len(args), desc="Processing files"))
        pool.close()
        pool.join()
    samples = []
    for result in results:
        if result[0] is not None:
            samples.append(result[0])
        for k, v in result[1].items():
            removal_reasons[k] = removal_reasons.get(k, 0) + v
    if samples:
        final_df = pd.concat(samples, ignore_index=True)
    else:
        final_df = pd.DataFrame()
    total_samples = len(final_df)
    return final_df, total_samples, removal_reasons

In [ ]:
#| export
def _open_zarr_fast(zarr_file):
    # Try consolidated metadata for faster attribute access
    try:
        return zarr.open_consolidated(zarr_file, mode='r')
    except Exception:
        return zarr.open(zarr_file, mode='r')


def _process_file_group(args):
    """Process all samples from a single file - designed for multiprocessing"""
    zarr_file, group_df, channels, channel_quality_functions, require_all_channels, frequency, sample_frequency_key = args
    
    removal_reasons = {"0":0, "1":0, "2":0, "3":0, "4":0}
    
    try:
        root_grp = _open_zarr_fast(zarr_file)
    except Exception:
        print(f"Could not open zarr file: {zarr_file}", flush=True)
        return None, removal_reasons

    avail_channels = list(root_grp.array_keys())
    updated_channels = [
        next((x for x in ch if x in avail_channels), None) if isinstance(ch, list)
        else ch if ch in avail_channels else None
        for ch in channels
    ]
    valid_channels = [ch for ch in updated_channels if ch is not None]
    
    if require_all_channels and len(valid_channels) != len(channels):
        removal_reasons["1"] += len(group_df)
        del root_grp
        return None, removal_reasons
    elif not valid_channels:
        removal_reasons["2"] += len(group_df)
        del root_grp
        return None, removal_reasons

    # Read all channels once into memory
    channel_data_dict = {}
    for channel in updated_channels:
        if channel is not None:
            signal = root_grp[channel][:]
            signal_length = len(signal)
            signal_all_nan = np.isnan(signal).all()
            if signal_length <= 1 or signal_all_nan:
                removal_reasons["2"] += len(group_df)
                del root_grp
                return None, removal_reasons
            channel_data_dict[channel] = signal

    # Process quality checks for each channel
    for channel in updated_channels:
        if channel is not None:
            func = channel_quality_functions.get(channel, None) if channel_quality_functions is not None else None
            if func is None:
                group_df[f'{channel}_quality'] = True
                continue
                
            channel_frequency = root_grp[channel].attrs.get(sample_frequency_key, None)
            if channel_frequency is None:
                channel_frequency = frequency

            
            start_indices = (group_df['start_idx'] / frequency * channel_frequency).astype(int).values
            end_indices = (group_df['end_idx'] / frequency * channel_frequency).astype(int).values
            
            data = channel_data_dict[channel]

            if channel_frequency != frequency:
                # resample data to match the sample frequency
                quality_results = []
                for start, end in zip(start_indices, end_indices):
                    segment = data[start:end]
                    resampled_segment = resample_waveform(segment, channel_frequency, frequency, is_spo2=False)
                    quality_result = func(resampled_segment, fs=frequency)
                    quality_results.append(quality_result)
            else:
                quality_results = [
                    func(data[start:end], fs=channel_frequency)
                    for start, end in zip(start_indices, end_indices)
                ]
            
            group_df[f'{channel}_quality'] = quality_results

    quality_cols = [f'{channel}_quality' for channel in updated_channels if channel is not None]
    group_df['all_channels_quality'] = group_df[quality_cols].all(axis=1)
    
    total_before = len(group_df)
    group_df = group_df.loc[group_df['all_channels_quality']].copy()
    removal_reasons["4"] += (total_before - len(group_df))

    if not group_df.empty:
        group_df['n_samples'] = len(group_df)
        removal_reasons["0"] += len(group_df)
        result = group_df
    else:
        result = None
        
    del root_grp
    return result, removal_reasons

def _process_file_segment(args, channels, channel_quality_functions, require_all_channels, frequency, sample_frequency_key):
    """Process all samples from a single file - designed for multiprocessing"""
    args = list(args)
    start_idx = args[0]
    end_idx = args[1]
    file_path = args[2]
    
    try:
        root_grp = _open_zarr_fast(file_path)
    except Exception:
        print(f"Could not open zarr file: {file_path}", flush=True)
        return None

    avail_channels = list(root_grp.array_keys())
    updated_channels = [
        next((x for x in ch if x in avail_channels), None) if isinstance(ch, list)
        else ch if ch in avail_channels else None
        for ch in channels
    ]
    valid_channels = [ch for ch in updated_channels if ch is not None]
    
    if require_all_channels and len(valid_channels) != len(channels):
        removal_reason = 1
        del root_grp
        return None, removal_reason
    elif not valid_channels:
        removal_reason = 2
        del root_grp
        return None, removal_reason

    # Process quality checks for each channel
    channel_quality_results = {}
    for channel in updated_channels:
        if channel is not None:
            func = channel_quality_functions.get(channel, None) if channel_quality_functions is not None else None
            if func is None:
                channel_quality_results[f'{channel}_quality'] = True
                continue
                
            channel_frequency = root_grp[channel].attrs.get(sample_frequency_key, None)
            if channel_frequency is None:
                channel_frequency = frequency

            start_idx_adj = int(int(start_idx) / frequency * int(channel_frequency))
            end_idx_adj = int(int(end_idx) / frequency * int(channel_frequency))

            data = root_grp[channel][start_idx_adj:end_idx_adj]

            quality_results = func(data, fs=channel_frequency)

            channel_quality_results[f'{channel}_quality'] = quality_results

    all_channels_quality = all(channel_quality_results[f'{channel}_quality'] for channel in updated_channels if channel is not None)
    
    if not all_channels_quality:
        removal_reason = 4
        del root_grp
        return None, removal_reason
    else:
        result = args
        removal_reason = 0
    del root_grp
    return result, removal_reason

def calculate_samples_forecast_df(
    outcome_df,
    file_col,
    outcome_val_col,
    outcome_time_col,
    outcome_duration_col,
    channels,
    forecast_window_sec,
    frequency,
    sample_seq_len_sec,
    channel_quality_functions=None,
    require_all_channels=True,
    infer_forecast_windows=False,
    sample_frequency_key='sampling_frequency'
):
    """Optimized version using pure multiprocessing"""
    
    removal_reasons = {"0":0, "1":0, "2":0, "3":0, "4":0}

    # Vectorized calculations
    sample_seq_len = int(sample_seq_len_sec * frequency)
    if isinstance(forecast_window_sec, list):
        forecast_offsets = [int(f * frequency) for f in forecast_window_sec]
    else:
        forecast_window_sec = [forecast_window_sec]
        forecast_offset = [int(forecast_window_sec * frequency)]

    outcome_start_idxs = (outcome_df[outcome_time_col].values * frequency).astype(int)
    outcome_end_idxs = (outcome_start_idxs + (outcome_df[outcome_duration_col].values * frequency).astype(int))
    all_dfs = []
    for forecast_offset, forecast_sec in zip(forecast_offsets, forecast_window_sec):
        start_idxs = (outcome_df[outcome_time_col].values * frequency - forecast_offset - sample_seq_len).astype(int)
        end_idxs = (outcome_df[outcome_time_col].values * frequency - forecast_offset).astype(int)

        df = pd.DataFrame({
            'start_idx': start_idxs,
            'end_idx': end_idxs,
            'file_path': outcome_df[file_col].values,
            'outcome_start_idx': outcome_start_idxs,
            'outcome_end_idx': outcome_end_idxs,
            'outcome_duration_sec': outcome_df[outcome_duration_col].values,
            f'outcome_val_{forecast_sec}sec': outcome_df[outcome_val_col].values
        })
        all_dfs.append(df)
    df = pd.concat(all_dfs, ignore_index=True)
    agg_dict = {col: 'first' for col in df.columns 
                if col not in ['start_idx', 'end_idx', 'file_path']}
    df = df.groupby(['start_idx', 'end_idx', 'file_path']).agg(agg_dict).reset_index()

    if infer_forecast_windows:
        for forecast_offset, forecast_sec in zip(forecast_offsets, forecast_window_sec):
            # check if the forecast window is within the outcome window, else assign to 0
            ## note that to force a 0, end outcome idx must be less than start outcome idx
            forecast_start_idxs = df['end_idx'] + forecast_offset
            #inferred_ = (forecast_start_idxs >= df['outcome_start_idx']) & (forecast_start_idxs <= df['outcome_end_idx'])
            df[f'outcome_val_{forecast_sec}sec'] = (forecast_start_idxs >= df['outcome_start_idx']) & (forecast_start_idxs <= df['outcome_end_idx']).astype(int)
    # Prefilter
    valid_mask = (df['start_idx'] >= 0) & (df['end_idx'] > 0)
    removal_reasons["3"] += (~valid_mask).sum()
    df = df.loc[valid_mask]

    # Group by file
    file_groups = [(zarr_file, group.copy()) for zarr_file, group in df.groupby('file_path', sort=False)]
    args = [
        (zarr_file, group, channels, channel_quality_functions, 
         require_all_channels, frequency, sample_frequency_key)
        for zarr_file, group in file_groups
    ] # it appears groups are faster

    #args = df.to_dict(orient='records')
    #args = list(map(lambda v: tuple(v.values()), args))

    
    fn = _process_file_group #partial(_process_file_segment, channels=channels, channel_quality_functions=channel_quality_functions, require_all_channels=require_all_channels, frequency=frequency, sample_frequency_key=sample_frequency_key)
    
    results = []
    with mp.Pool() as pool:
        for result, reason in tqdm.tqdm(
            pool.imap_unordered(fn, args),
            total=len(args),
            desc="Processing files"
        ):
            if result is not None:
                results.append(result)
            for k, v in reason.items():
                removal_reasons[k] = removal_reasons.get(k, 0) + v
                #removal_reasons[str(reason)] += 1

    if results:
        final_df = pd.concat(results, ignore_index=True) # pd.DataFrame(results)
    else:
        final_df = pd.DataFrame()

    total_samples = len(final_df)
    return final_df, total_samples, removal_reasons

# #| export
# def calculate_samples_forecast_df(
#     outcome_df,
#     file_col,
#     outcome_val_col,
#     outcome_time_col,
#     outcome_duration_col,
#     channels,
#     forecast_window_sec,
#     frequency,
#     sample_seq_len_sec,
#     channel_quality_functions=None,
#     require_all_channels=True,
#     infer_forecast_windows=False,
#     sample_frequency_key='sampling_frequency',
#     n_workers=None,
#     batch_size=50  # Process files in batches
# ):
#     """Hybrid approach: Process files in batches to avoid stalls"""
    
#     removal_reasons = {"1":0, "2":0, "3":0, "4":0, "5":0}

#     # Vectorized calculations
#     sample_seq_len = int(sample_seq_len_sec * frequency)
#     forecast_offset = int(forecast_window_sec * frequency)

#     start_idxs = (outcome_df[outcome_time_col].values * frequency - forecast_offset - sample_seq_len).astype(int)
#     end_idxs = (outcome_df[outcome_time_col].values * frequency - forecast_offset).astype(int)
#     outcome_start_idxs = (outcome_df[outcome_time_col].values * frequency).astype(int)
#     outcome_end_idxs = (outcome_start_idxs + (outcome_df[outcome_duration_col].values * frequency).astype(int))

#     valid_mask = (start_idxs >= 0) & (end_idxs > 0)
#     removal_reasons["3"] += (~valid_mask).sum()
    
#     df = pd.DataFrame({
#         'start_idx': start_idxs[valid_mask],
#         'end_idx': end_idxs[valid_mask],
#         'file_path': outcome_df[file_col].values[valid_mask],
#         'outcome_start_idx': outcome_start_idxs[valid_mask],
#         'outcome_end_idx': outcome_end_idxs[valid_mask],
#         'outcome_duration_sec': outcome_df[outcome_duration_col].values[valid_mask],
#         f'outcome_val_{forecast_window_sec}sec': outcome_df[outcome_val_col].values[valid_mask]
#     })

#     if df.empty:
#         return pd.DataFrame(), 0, removal_reasons

#     df = df.sort_values('file_path')
#     file_groups = [(zarr_file, group.reset_index(drop=True)) for zarr_file, group in df.groupby('file_path', sort=False)]
    
#     args_list = [
#         (zarr_file, group, channels, channel_quality_functions, 
#          require_all_channels, frequency, sample_frequency_key)
#         for zarr_file, group in file_groups
#     ]
    
#     if n_workers is None:
#         n_workers = mp.cpu_count()
    
#     # Process in batches to avoid resource exhaustion
#     all_results = []
#     n_batches = (len(args_list) + batch_size - 1) // batch_size
    
#     for batch_idx in range(n_batches):
#         start_idx = batch_idx * batch_size
#         end_idx = min((batch_idx + 1) * batch_size, len(args_list))
#         batch_args = args_list[start_idx:end_idx]
        
#         print(f"Processing batch {batch_idx+1}/{n_batches} ({len(batch_args)} files)", flush=True)
        
#         # Process batch with fresh pool each time
#         with mp.Pool(processes=n_workers) as pool:
#             batch_results = list(tqdm.tqdm(
#                 pool.imap_unordered(_process_file_group_robust, batch_args),
#                 total=len(batch_args),
#                 desc=f"Batch {batch_idx+1}/{n_batches}"
#             ))
#             pool.close()
#             pool.join()
        
#         for result, reasons in batch_results:
#             if result is not None:
#                 all_results.append(result)
#             for k, v in reasons.items():
#                 removal_reasons[k] = removal_reasons.get(k, 0) + v
        
#         # Small delay between batches
#         time.sleep(0.5)

#     if all_results:
#         final_df = pd.concat(all_results, ignore_index=True, copy=False)
#     else:
#         final_df = pd.DataFrame()

#     return final_df, len(final_df), removal_reasons

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()